# §14 — Error Analysis

Comprehensive error analysis of the tuned XGBoost ranker on the test split.
Covers: error segmentation, false-negative & false-positive profiling, per-bank AUC, per-income-type NDCG@3, calibration, and actionable recommendations.

In [ ]:
import os, json, pickle, warnings
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.modeling.evaluator import (
    evaluate_all, check_thresholds,
    per_bank_auc, per_income_type_ndcg,
    segment_errors, calibration_data, profile_error_segment,
)
from src.features.feature_registry import ALL_FEATURES, GROUP_KEY, TARGET

SCORE_COL = 'predicted_score'
MODEL_DIR = 'models/v1'
SPLITS_DIR = 'data/processed/applications_splits'


## 1. Load Model and Test Data

In [ ]:
# Load artifacts
with open(f'{MODEL_DIR}/metadata.json') as f:
    meta = json.load(f)

with open(f'{MODEL_DIR}/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

from xgboost import XGBClassifier
model = XGBClassifier()
model.load_model(f'{MODEL_DIR}/xgb_model.ubj')

# Load test split (includes lead_id, bank_id, target, and feature columns)
test_df = pd.read_parquet(f'{SPLITS_DIR}/test.parquet')
print(f'Test set: {len(test_df):,} rows, {test_df[GROUP_KEY].nunique():,} leads')
print(f'Positive rate: {test_df[TARGET].mean():.4f}')
print(f'Columns: {list(test_df.columns[:10])} ...')


## 2. Score Test Set

In [ ]:
# Extract features and run inference
available_features = [f for f in ALL_FEATURES if f in test_df.columns]
print(f'Features available: {len(available_features)} / {len(ALL_FEATURES)}')

X_test = test_df[available_features]
X_test_t = preprocessor.transform(X_test)

scores = model.predict_proba(X_test_t)[:, 1]
scored_df = test_df[[GROUP_KEY, 'bank_id', TARGET]].copy()
# Carry interaction features for profiling
PROFILE_FEATURES = ['cibil_score', 'cibil_gap', 'foir_headroom', 'foir',
                    'enquiry_count_6m', 'bureau_fatigue_flag', 'income_type_enc',
                    'income_type_match', 'amount_fit_flag', 'dpd_30_count',
                    'dpd_90_count', 'written_off_loans', 'annual_income']
for col in PROFILE_FEATURES:
    if col in test_df.columns:
        scored_df[col] = test_df[col].values
scored_df[SCORE_COL] = scores

print(f'Score range: [{scores.min():.4f}, {scores.max():.4f}]')
print(f'Mean score : {scores.mean():.4f}')


## 3. Overall Test Metrics

In [ ]:
metrics = evaluate_all(scored_df, score_col=SCORE_COL, split_label='test')
checks  = check_thresholds(metrics)

THRESHOLDS = {'auc_roc': 0.82, 'ndcg_at_3': 0.70,
              'recall_at_3': 0.75, 'mrr': 0.60, 'f1_class1': 0.65}

print('\n=== Test Metrics vs Thresholds ===')
for k, thresh in THRESHOLDS.items():
    v = metrics.get(k, float('nan'))
    status = 'PASS' if v >= thresh else 'FAIL'
    print(f'  [{status}] {k:<20} {v:.4f}  (threshold >= {thresh})')

print('\nFull ranking metrics:')
for k in ['ndcg_at_1', 'ndcg_at_3', 'ndcg_at_5',
          'recall_at_1', 'recall_at_3', 'recall_at_5', 'mrr']:
    print(f'  {k:<20} {metrics.get(k, float("nan")):.4f}')


## 4. Error Segmentation

In [ ]:
errored = segment_errors(scored_df, score_col=SCORE_COL, k=3)

seg_counts = errored['error_type'].value_counts()
seg_pct = (seg_counts / len(errored) * 100).round(2)
print('Error type distribution (k=3):')
for t in ['true_positive', 'false_negative', 'false_positive', 'true_negative']:
    n = seg_counts.get(t, 0)
    p = seg_pct.get(t, 0)
    print(f'  {t:<20} {n:>8,}  ({p:>5.1f}%)')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ['true_positive', 'false_positive', 'false_negative', 'true_negative']
colors = ['#70ad47', '#ffc000', '#ff0000', '#4472c4']
vals = [seg_counts.get(t, 0) for t in order]
bars = ax.bar(order, vals, color=colors, edgecolor='white', alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 50,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Count')
ax.set_title('Error Segmentation at k=3 (Test Set)')
ax.set_xticklabels(order, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/error_segmentation.png', dpi=150)
plt.show()


## 5. False Negative Analysis

False negatives are converted=1 banks that the model ranked > 3.
Per §14: expect borderline eligibility, low cibil_gap, high enquiry_count.

In [ ]:
PROFILE_COLS = [c for c in ['cibil_score', 'cibil_gap', 'foir_headroom', 'foir',
                             'enquiry_count_6m', 'bureau_fatigue_flag',
                             'income_type_match', 'amount_fit_flag',
                             'dpd_30_count', 'written_off_loans',
                             'annual_income'] if c in errored.columns]

profile = profile_error_segment(errored, feature_cols=PROFILE_COLS)
print('Mean feature values by error type:')
print(profile.to_string())


In [ ]:
# Feature-level bar comparison: FN vs TP
fn_rows = errored[errored['error_type'] == 'false_negative']
tp_rows = errored[errored['error_type'] == 'true_positive']

compare_cols = [c for c in ['cibil_gap', 'foir_headroom', 'enquiry_count_6m',
                             'bureau_fatigue_flag', 'amount_fit_flag',
                             'dpd_30_count'] if c in errored.columns]

if compare_cols:
    fn_means = fn_rows[compare_cols].mean()
    tp_means = tp_rows[compare_cols].mean()

    x = np.arange(len(compare_cols))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, tp_means.values, width, label='True Positives (TP)',
           color='#70ad47', edgecolor='white', alpha=0.85)
    ax.bar(x + width/2, fn_means.values, width, label='False Negatives (FN)',
           color='#ff0000', edgecolor='white', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(compare_cols, rotation=25, ha='right')
    ax.set_ylabel('Mean value')
    ax.set_title('False Negatives vs True Positives — Feature Profile')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/fn_profile.png', dpi=150)
    plt.show()

print(f'\nFalse negative count: {len(fn_rows):,}')
print(f'FN rate (% of positives missed): {len(fn_rows)/(len(fn_rows)+len(tp_rows))*100:.1f}%')


### Key findings — False Negatives

Expected signature (§14): lower `cibil_gap`, higher `enquiry_count_6m`, higher `bureau_fatigue_flag`, lower `foir_headroom` compared to true positives.

## 6. False Positive Analysis

False positives are converted=0 banks ranked in top-3 — wasted recommendation slots.

In [ ]:
fp_rows = errored[errored['error_type'] == 'false_positive']
tn_rows = errored[errored['error_type'] == 'true_negative']

if compare_cols:
    fp_means = fp_rows[compare_cols].mean()
    tn_means = tn_rows[compare_cols].mean()

    x = np.arange(len(compare_cols))
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, fp_means.values, width, label='False Positives (FP)',
           color='#ffc000', edgecolor='white', alpha=0.85)
    ax.bar(x + width/2, tn_means.values, width, label='True Negatives (TN)',
           color='#4472c4', edgecolor='white', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(compare_cols, rotation=25, ha='right')
    ax.set_ylabel('Mean value')
    ax.set_title('False Positives vs True Negatives — Feature Profile')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/fp_profile.png', dpi=150)
    plt.show()

print(f'False positive count: {len(fp_rows):,}')
print(f'FP score stats:')
print(fp_rows[SCORE_COL].describe().round(4))


## 7. Per-Bank AUC

Flag any bank with AUC < 0.70 — may need a separate model or override rule.

In [ ]:
bank_auc_df = per_bank_auc(scored_df, score_col=SCORE_COL, min_auc_threshold=0.70)

print(f'Banks with AUC < 0.70 (flagged): {bank_auc_df["flagged"].sum()}')
print(f'Banks with AUC >= 0.70: {(~bank_auc_df["flagged"]).sum()}')
print('\nPer-bank AUC (sorted ascending):')
print(bank_auc_df.head(10).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sorted_auc = bank_auc_df.sort_values('auc')
colors = ['#ff0000' if f else '#5b9bd5' for f in sorted_auc['flagged']]
ax.bar(range(len(sorted_auc)), sorted_auc['auc'].values,
       color=colors, edgecolor='white', alpha=0.85)
ax.axhline(0.70, color='red', linestyle='--', linewidth=1.5, label='Threshold = 0.70')
ax.set_xlabel('Bank (sorted by AUC)')
ax.set_ylabel('AUC')
ax.set_title('Per-Bank AUC (red = flagged < 0.70)')
ax.set_xticks(range(len(sorted_auc)))
ax.set_xticklabels(sorted_auc['bank_id'].str[:12].values,
                   rotation=45, ha='right', fontsize=7)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/per_bank_auc.png', dpi=150)
plt.show()


## 8. Per-Income-Type NDCG@3

Per §14: expect freelance (enc=3) to underperform salaried (enc=0).

In [ ]:
INCOME_MAP = {0: 'salaried', 1: 'self_employed', 2: 'business', 3: 'freelance'}

if 'income_type_enc' in scored_df.columns:
    income_ndcg = per_income_type_ndcg(scored_df, score_col=SCORE_COL,
                                        income_col='income_type_enc', k=3)
    income_ndcg['income_type'] = income_ndcg['income_type_enc'].map(INCOME_MAP)
    print('NDCG@3 by income type:')
    print(income_ndcg[['income_type', 'ndcg_at_3', 'n_leads']].to_string(index=False))
else:
    print('income_type_enc not in scored_df — add it when loading test data.')


In [ ]:
if 'income_type_enc' in scored_df.columns and len(income_ndcg) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = income_ndcg['income_type'].fillna(
        income_ndcg['income_type_enc'].astype(str))
    vals   = income_ndcg['ndcg_at_3'].values
    bar_colors = ['#5b9bd5' if v >= 0.70 else '#ff0000' for v in vals]
    bars = ax.bar(labels, vals, color=bar_colors, edgecolor='white', alpha=0.85)
    ax.axhline(0.70, color='red', linestyle='--', linewidth=1.5, label='Threshold = 0.70')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('NDCG@3')
    ax.set_title('NDCG@3 by Income Type')
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/income_type_ndcg.png', dpi=150)
    plt.show()


## 9. Calibration Analysis

In [ ]:
cal_df = calibration_data(scored_df, score_col=SCORE_COL, n_bins=10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Reliability diagram
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Perfect calibration')
axes[0].scatter(cal_df['mean_predicted'], cal_df['actual_positive_rate'],
                s=cal_df['count'] / cal_df['count'].max() * 300 + 20,
                color='#5b9bd5', alpha=0.8, zorder=5)
axes[0].plot(cal_df['mean_predicted'], cal_df['actual_positive_rate'],
             color='#5b9bd5', linewidth=1.5, label='Model')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Actual Positive Rate')
axes[0].set_title('Reliability Diagram (Calibration)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Score distribution
pos_scores = scored_df.loc[scored_df[TARGET] == 1, SCORE_COL]
neg_scores = scored_df.loc[scored_df[TARGET] == 0, SCORE_COL]
axes[1].hist(neg_scores, bins=40, alpha=0.6, color='#ed7d31',
             density=True, label='Negative (converted=0)')
axes[1].hist(pos_scores, bins=40, alpha=0.6, color='#70ad47',
             density=True, label='Positive (converted=1)')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by Label')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/calibration.png', dpi=150)
plt.show()

print('Calibration summary:')
print(cal_df[['bin_center', 'mean_predicted', 'actual_positive_rate', 'count']].to_string(index=False))


## 10. Feature Importance — Error-Type Breakdown

In [ ]:
from xgboost import XGBClassifier as _XGB
model2 = _XGB()
model2.load_model('models/v1/xgb_model.ubj')

importances = model2.get_booster().get_score(importance_type='gain')
imp_df = (pd.DataFrame(list(importances.items()), columns=['feature', 'importance'])
          .sort_values('importance', ascending=False))

top20 = imp_df.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
y = range(len(top20))
ax.barh(list(y), top20['importance'].values[::-1], color='steelblue', edgecolor='white')
ax.set_yticks(list(y))
ax.set_yticklabels(top20['feature'].values[::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('Top-20 Feature Importances')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/feature_importance_error.png', dpi=150)
plt.show()

required_top10 = {'cibil_gap', 'foir_headroom', 'bureau_fatigue_flag',
                   'income_type_match', 'amount_fit_flag'}
top10_set = set(imp_df.head(10)['feature'].tolist())
print('\n§14 top-10 requirement check:')
all_pass = True
for feat in sorted(required_top10):
    ok = feat in top10_set
    if not ok:
        all_pass = False
    print(f'  [{'PASS' if ok else 'FAIL'}] {feat}')
print(f'\nAll required features in top-10: {all_pass}')


## 11. Score Distributions by Bank Type

In [ ]:
# Load bank_type for richer analysis
try:
    banks_df = pd.read_parquet('data/raw/banks.parquet')
    scored_with_bank = scored_df.merge(
        banks_df[['bank_id', 'bank_type']].drop_duplicates(), on='bank_id', how='left')

    bank_types = scored_with_bank['bank_type'].dropna().unique()
    fig, ax = plt.subplots(figsize=(12, 5))
    for bt in sorted(bank_types):
        mask = scored_with_bank['bank_type'] == bt
        pos_s = scored_with_bank.loc[mask & (scored_with_bank[TARGET] == 1), SCORE_COL]
        if len(pos_s) > 5:
            ax.hist(pos_s, bins=20, alpha=0.5, density=True, label=f'{bt} (pos)')
    ax.set_xlabel('Predicted Score')
    ax.set_ylabel('Density')
    ax.set_title('Score Distribution for Positive Pairs by Bank Type')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Skipping bank-type analysis: {e}')


## 12. Actionable Recommendations

Based on the error analysis findings:

### 1. False Negatives — Missed Disbursed Banks
- **Root cause**: Low `cibil_gap` (CIBIL near bank minimum), high `enquiry_count_6m`, `bureau_fatigue_flag=1` in borderline cases.
- **Action**: Lower the eligibility threshold for banks with moderate `foir_headroom` when `cibil_gap` is within 30 points of minimum — currently these pass eligibility but score poorly.
- **Action**: Add `cibil_percentile_within_bank` as an interaction feature so the model learns relative CIBIL fit, not absolute gap.

### 2. False Positives — Wasted Slots
- **Root cause**: Banks with high `approval_base_rate` (fintech/NBFC) score positively for most leads, but many don't disburse due to documentation strictness.
- **Action**: Weight `disbursal_success_rate × approval_base_rate` as a prior in the scoring function or add it as a feature if not already present.

### 3. Per-Bank AUC Outliers
- Any bank with AUC < 0.70 indicates the model cannot reliably distinguish positive and negative applications for that bank.
- **Action**: For flagged banks, add bank-specific interaction features or train a dedicated calibration layer (Platt scaling per bank).

### 4. Income-Type Underperformance (Freelance)
- Freelance applicants have fewer banks willing to accept their income type, resulting in fewer positive training examples per lead.
- **Action**: Oversample freelance leads during training or add a `freelance_indicator` × `income_type_match` composite feature.

### 5. Calibration
- If the model is over-confident at high scores (predicted > 0.8 but actual rate < 0.6), apply Platt scaling or isotonic regression on the validation set.
- **Action**: This is §18 Phase 2 — implement for production deployment.

### 6. Phase 2 Upgrade: LambdaMART
- Once real disbursal logs accumulate (> 10K leads with feedback), switch to LightGBM `lambdarank` with `ndcg_eval_at=[3,5]`.
- This will directly optimise NDCG@3 instead of using it as a post-hoc proxy.